# PaperNoteAgent：可核验的论文精读笔记助手

> 输入一篇论文（arXiv 编号或本地 PDF），生成一份**每条要点都附原文证据、并由程序逐句核对**的中文精读笔记。

作者：[@hahahha123321](https://github.com/hahahha123321)

日期：2026-09-30

**适合读者**
- 读英文论文吃力、又担心 AI 总结“编造内容”的同学
- 想学习“生成 Agent + 审核 Agent + 程序校验”协作方式的 HelloAgents 学习者

**前置条件**
- Python 3.10+，已安装 `requirements.txt`
- 一个 OpenAI 兼容的 LLM API（`.env` 中配置）
- 第 10 节“自检”不需要 API，可以先运行它

**学习目标**
- 用 pypdf 读取论文并做上下文裁剪（第 9 章 上下文工程）
- 用 Pydantic 约束模型输出的 JSON（第 13 章）
- 两个 `SimpleAgent` 协作：ReaderAgent 写笔记，CheckerAgent 审核
- 用普通 Python 逐句核对“原文证据”，算出证据命中率（第 12 章 评估）

## 学习路线

1. 导入与环境
2. 整体架构
3. 读取论文：arXiv 下载 / 本地 PDF → 提取文字 → 清洗 → 限长
4. 定义笔记的数据结构
5. 程序核对原文证据
6. 解析模型返回的 JSON
7. 两个 Agent 的提示词
8. 调用预算与格式修复
9. 编排完整流程
10. 生成 Markdown 笔记
11. 自检（不调用模型）
12. 运行真实论文
13. 总结与展望

## 1. 导入与环境

In [1]:
from __future__ import annotations

import json
import re
import unicodedata
import urllib.request
from collections import Counter
from pathlib import Path
from typing import Literal

from dotenv import load_dotenv
from hello_agents import HelloAgentsLLM, SimpleAgent
from pydantic import BaseModel, Field, ValidationError
from pypdf import PdfReader

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

# hello-agents 不会自动读取 .env，需要手动加载
load_dotenv(PROJECT_ROOT / ".env")
print(f"项目目录: {PROJECT_ROOT.name}")

项目目录: hahahha123321-PaperNoteAgent


## 2. 整体架构

```text
arXiv 编号 / 本地 PDF
  → [Python] 下载、提取文字、去掉参考文献、清洗、限长
  → [ReaderAgent] 生成精读笔记 JSON：每条要点 = 中文概括 + 英文原句证据
  → [Python] 逐条核对证据是否真的出现在原文中
  → [CheckerAgent] 审核要点与证据是否相符、有无遗漏
  → 不通过则修正一次（全程最多 4 次模型调用）
  → [Python] 生成 JSON + Markdown 笔记
```

设计原则：**确定性的工作交给 Python，理解和判断交给 Agent。**
“证据是否出现在原文”是一个可以直接查的问题，所以不交给模型判断，而是由程序逐字核对。

编排思路参考了共创项目 `Henry2513-MeetingActionAgent`。

## 3. 读取论文

- 输入是 arXiv 编号（如 `2210.03629`）时，自动下载 PDF 到 `data/`；已经下载过就直接用缓存。
- 输入是 `.pdf` 路径时，直接读取本地文件。
- 每页重复出现的页眉和单独一行的页码会打断正文句子，先去掉。
- 参考文献对理解论文帮助不大、却很占篇幅，所以截掉。
- 论文太长时只保留前 `MAX_PAPER_CHARS` 个字符，并在笔记中注明。

In [2]:
MAX_PAPER_CHARS = 60_000
ARXIV_ID_PATTERN = re.compile(r"^(?:arxiv:)?(\d{4}\.\d{4,5})(?:v\d+)?$", re.IGNORECASE)
REFERENCES_HEADING = re.compile(
    r"^\s*(?:\d+\.?\s*)?(?:references|bibliography)\s*$", re.IGNORECASE | re.MULTILINE
)


# 表示清洗后的论文正文。
class PaperText(BaseModel):
    source: str
    text: str
    original_chars: int
    truncated: bool


# 识别 arXiv 编号（允许 arXiv: 前缀和 v2 之类的版本号），识别不了返回 None。
def parse_arxiv_id(source: str) -> str | None:
    match = ARXIV_ID_PATTERN.match(source.strip())
    return match.group(1) if match else None


# 从 arXiv 下载 PDF；已经下载过就直接使用本地缓存。
def download_arxiv_pdf(arxiv_id: str) -> Path:
    pdf_path = DATA_DIR / f"{arxiv_id}.pdf"
    if pdf_path.exists():
        return pdf_path
    request = urllib.request.Request(
        f"https://arxiv.org/pdf/{arxiv_id}",
        headers={"User-Agent": "PaperNoteAgent (Hello-Agents graduation project)"},
    )
    with urllib.request.urlopen(request, timeout=60) as response:
        pdf_path.write_bytes(response.read())
    return pdf_path


# 把用户输入（arXiv 编号或本地路径）解析成一个 PDF 文件路径。
def resolve_pdf(source: str) -> Path:
    arxiv_id = parse_arxiv_id(source)
    if arxiv_id:
        return download_arxiv_pdf(arxiv_id)
    path = Path(source.strip()).expanduser()
    if not path.is_absolute():
        path = PROJECT_ROOT / path
    if path.suffix.lower() != ".pdf":
        raise ValueError(f"只支持 arXiv 编号或 .pdf 文件：{source}")
    if not path.exists():
        raise FileNotFoundError(f"找不到文件：{path}")
    return path


# 去掉页眉和页码：在一半以上页面重复出现的首行视为页眉，只含数字的末行视为页码。
def strip_page_furniture(pages: list[str]) -> list[str]:
    first_lines = Counter(page.strip().split("\n", 1)[0] for page in pages if page.strip())
    header, count = first_lines.most_common(1)[0] if first_lines else ("", 0)
    has_header = len(pages) >= 4 and count > len(pages) / 2
    cleaned = []
    for page in pages:
        lines = page.strip().split("\n")
        if has_header and lines[0] == header:
            lines = lines[1:]
        if lines and lines[-1].strip().isdigit():
            lines = lines[:-1]
        cleaned.append("\n".join(lines))
    return cleaned


# 截掉参考文献及其后的附录；只认位于全文 30% 之后的标题行，避免误伤目录。
def strip_references(raw_text: str) -> str:
    for match in REFERENCES_HEADING.finditer(raw_text):
        if match.start() > len(raw_text) * 0.3:
            return raw_text[: match.start()]
    return raw_text


# 清洗 PDF 文字：统一字符（如连字 ﬁ → fi）、接回行尾断开的单词、压缩空白。
def clean_text(raw_text: str) -> str:
    text = unicodedata.normalize("NFKC", raw_text)
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


# 读取论文：定位 PDF → 提取文字 → 去页眉页码 → 去参考文献 → 清洗 → 限长。
def load_paper(source: str) -> PaperText:
    pdf_path = resolve_pdf(source)
    reader = PdfReader(pdf_path)
    pages = strip_page_furniture([page.extract_text() or "" for page in reader.pages])
    raw_text = "\n".join(pages)
    text = clean_text(strip_references(raw_text))
    if len(text) < 500:
        raise ValueError("提取到的文字太少，可能是扫描版（图片）PDF，pypdf 无法识别")
    return PaperText(
        source=source.strip(),
        text=text[:MAX_PAPER_CHARS],
        original_chars=len(text),
        truncated=len(text) > MAX_PAPER_CHARS,
    )

## 4. 笔记的数据结构

每条要点（`Claim`）都由两部分组成：
- `point`：中文概括，方便阅读
- `evidence`：从原文**逐字复制**的英文句子，用来核对

`Field(description=...)` 里的说明会出现在 JSON Schema 中，模型能直接看到每个字段该填什么。

In [3]:
# 一条要点：中文概括 + 原文证据。
class Claim(BaseModel):
    point: str = Field(min_length=1, description="用中文概括的一条要点")
    evidence: str = Field(
        min_length=1,
        description="从论文原文逐字复制的一句连续英文原句，不翻译、不改写、不使用省略号",
    )


# ReaderAgent 输出的精读笔记内容。
class PaperNote(BaseModel):
    title: str = Field(min_length=1, description="论文的英文标题，按正常大小写书写（PDF 提取的标题可能全是大写或夹有多余空格）")
    one_sentence_summary: str = Field(min_length=1, description="一句话中文总结")
    keywords: list[str] = Field(default_factory=list, description="3~5 个关键词")
    research_question: list[Claim] = Field(min_length=1, description="论文要解决的问题")
    method: list[Claim] = Field(min_length=1, description="核心方法")
    experiments: list[Claim] = Field(default_factory=list, description="主要实验结论")
    limitations: list[Claim] = Field(
        default_factory=list, description="论文自己承认的局限；原文没写就返回空列表"
    )


# CheckerAgent 的审核结论。
class CheckResult(BaseModel):
    passed: bool
    issues: list[str] = Field(
        default_factory=list,
        description="其他问题；不要重复 unsupported_points 和 missing_points 中已有的内容",
    )
    unsupported_points: list[str] = Field(
        default_factory=list, description="与证据意思不符或夸大的要点"
    )
    missing_points: list[str] = Field(
        default_factory=list, description="遗漏的核心贡献或最主要的实验结论；次要细节不要写在这里"
    )
    revision_advice: list[str] = Field(
        default_factory=list, description="可选的改进建议，包括可以补充的次要细节；不影响是否通过"
    )


# 最终报告：笔记 + 程序核对结果 + 审核状态。
class NoteReport(BaseModel):
    source: str
    truncated: bool
    note: PaperNote
    evidence_hit_rate: float
    missing_evidence: list[str] = Field(default_factory=list)
    review_status: Literal["passed", "needs_manual_review"]
    review_issues: list[str] = Field(default_factory=list)
    model_calls: int

## 5. 程序核对原文证据

PDF 提取出的文字常常有多余空格、换行断词、大小写差异，直接用 `in` 比较会误判。
所以比较之前先把两边都“压扁”：统一字符、转小写、只保留字母和数字（以及汉字）。

太短的证据（压扁后少于 20 个字符，比如只写了 `ReAct`）几乎在任何论文里都能找到，没有核对意义，也算作不合格。

还有一种情况：一句话正好跨页，中间被脚注、图表标题隔开。这时整句找不到，但前半段和后半段都在原文里，而且离得很近。
`evidence_found` 允许证据被这样“打断一次”：两段都至少 20 个字符，间隔不超过 300 个字符，就算命中。

In [4]:
MIN_EVIDENCE_CHARS = 20
MAX_GAP_CHARS = 300


# 比对前的“压扁”：统一字符和大小写，只保留字母、数字和汉字。
def squash(text: str) -> str:
    text = unicodedata.normalize("NFKC", text).lower()
    return re.sub(r"[^0-9a-z\u4e00-\u9fff]+", "", text)


# 判断一条（已压扁的）证据是否出现在（已压扁的）原文中；允许它被脚注等内容从中间打断一次。
def evidence_found(needle: str, haystack: str) -> bool:
    if len(needle) < MIN_EVIDENCE_CHARS:
        return False
    if needle in haystack:
        return True
    # 从长到短，找出能在原文中找到的最长前半段，再看后半段是否在它之后不远处出现
    for cut in range(len(needle) - MIN_EVIDENCE_CHARS, MIN_EVIDENCE_CHARS - 1, -1):
        start = haystack.find(needle[:cut])
        if start != -1:
            tail = needle[cut:]
            tail_from = start + cut
            return haystack.find(tail, tail_from, tail_from + MAX_GAP_CHARS + len(tail)) != -1
    return False


# 列出笔记中的所有要点。
def all_claims(note: PaperNote) -> list[Claim]:
    return note.research_question + note.method + note.experiments + note.limitations


# 逐条检查 evidence 是否真的出现在原文中，返回命中率和找不到的证据。
def check_evidence(note: PaperNote, paper_text: str) -> tuple[float, list[str]]:
    haystack = squash(paper_text)
    claims = all_claims(note)
    missing = [claim.evidence for claim in claims if not evidence_found(squash(claim.evidence), haystack)]
    hit_rate = 1 - len(missing) / len(claims)
    return hit_rate, missing

## 6. 解析模型返回的 JSON

模型有时会把 JSON 包在 ```` ```json ```` 代码围栏里，或者前后加一句解释。
这里截取第一个 `{` 到最后一个 `}` 之间的内容，再交给 Pydantic 校验。

In [5]:
# 从模型回复中截取最外层 JSON 对象。
def extract_json_object(text: str) -> dict:
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end < start:
        raise ValueError("模型回复中没有完整的 JSON 对象")
    return json.loads(text[start : end + 1])


# 把模型回复解析并校验为指定的 Pydantic 模型。
def parse_model_response(text: str, model_type: type[BaseModel]) -> BaseModel:
    return model_type.model_validate(extract_json_object(text))

## 7. 两个 Agent 的提示词

- **ReaderAgent**：只写原文支持的内容；证据必须逐字复制；原文没写的局限性不许编。
- **CheckerAgent**：程序已经核对过“证据在不在原文里”，它只需要关注“要点和证据意思对不对得上”和“有没有漏掉核心内容”。
  笔记是摘要而不是全文复述，所以只有遗漏核心贡献才算不通过，次要细节只作为建议。

In [6]:
READER_SYSTEM_PROMPT = """你是严谨的学术论文精读助手。
只使用论文原文明确支持的信息，不得补充外部知识或猜测。
每条要点的 point 用中文概括；evidence 必须从原文逐字复制一句连续的英文原句，不得翻译、改写、拼接或使用省略号。
如果论文没有讨论局限性，limitations 返回空列表，不要自行编造。
只返回符合用户给定 JSON Schema 的 JSON，不要返回 Markdown 或额外解释。"""

CHECKER_SYSTEM_PROMPT = """你是独立的论文笔记审核员。
逐条检查每个要点 point 是否被它的 evidence 支持，有没有曲解或夸大原文。
精读笔记是摘要，不是全文复述：只有遗漏了论文的核心贡献或最主要的实验结论，才写入 missing_points。
次要细节、具体数字、原因分析等可以补充的内容只写进 revision_advice，不影响是否通过。
passed 的标准：没有曲解或夸大的要点，没有遗漏核心贡献，并且程序核对中没有找不到的证据。
证据是否出现在原文中已经由程序核对过，结果会附在最后，请一并考虑。
不能因为文字通顺就判定通过，也不能使用论文以外的信息。
只返回符合用户给定 JSON Schema 的 JSON，不要返回 Markdown 或额外解释。"""

## 8. 调用预算与格式修复

- 整个流程最多调用 4 次模型，防止两个 Agent 无限循环、浪费费用。
- 每次调用前执行 `clear_history()`：`SimpleAgent.run()` 会把每轮对话存进历史，下一轮再全部发给模型。
  论文很长，不清空的话，第二次调用会把整篇论文再发一遍，还可能被上一轮的回答带偏。
- JSON 第一次解析失败时，允许请求一次“只修格式”；修复同样计入预算。
- `temperature=0.2`：HelloAgentsLLM 默认是 0.7，做信息提取需要更稳定的输出。

In [7]:
class CallBudget:
    # 初始化模型调用次数上限。
    def __init__(self, maximum: int = 4) -> None:
        self.maximum = maximum
        self.used = 0

    # 剩余可用的调用次数。
    @property
    def remaining(self) -> int:
        return self.maximum - self.used

    # 在次数限制内调用一次 Agent；调用前清空历史，让每次调用都是独立的一轮对话。
    def run(self, agent, prompt: str) -> str:
        if self.remaining <= 0:
            raise RuntimeError(f"已达到 {self.maximum} 次模型调用上限")
        self.used += 1
        agent.clear_history()
        return agent.run(prompt)


# 调用 Agent 并把回复解析为指定的数据模型；格式错误时请求修复一次。
def run_structured(agent, prompt: str, model_type: type[BaseModel], budget: CallBudget) -> BaseModel:
    schema = json.dumps(model_type.model_json_schema(), ensure_ascii=False)
    raw_response = budget.run(agent, f"{prompt}\n\n必须遵循以下 JSON Schema：\n{schema}")
    try:
        return parse_model_response(raw_response, model_type)
    except ValueError as error:
        repair_prompt = (
            "上一次回复无法通过 JSON 校验。不要改变内容含义，只修复格式。\n"
            f"校验错误：{error}\n"
            f"原回复：\n{raw_response}\n"
            f"目标 Schema：\n{schema}\n"
            "只返回修复后的 JSON。"
        )
        repaired_response = budget.run(agent, repair_prompt)
        try:
            return parse_model_response(repaired_response, model_type)
        except ValueError as second_error:
            raise RuntimeError(f"修复后仍无法通过 JSON 校验：{second_error}") from second_error


# 创建 ReaderAgent 和 CheckerAgent（共用同一个 LLM）。
def build_agents() -> tuple[SimpleAgent, SimpleAgent]:
    llm = HelloAgentsLLM(temperature=0.2, max_tokens=4096)
    reader = SimpleAgent(name="ReaderAgent", llm=llm, system_prompt=READER_SYSTEM_PROMPT)
    checker = SimpleAgent(name="CheckerAgent", llm=llm, system_prompt=CHECKER_SYSTEM_PROMPT)
    return reader, checker

## 9. 编排完整流程

| 情况 | 模型调用次数 | 结果 |
|---|---|---|
| 初稿证据全部命中，审核通过 | 2 | `passed` |
| 初稿有问题 → 修正后通过 | 4 | `passed` |
| 修正后仍有问题 | 4 | `needs_manual_review` |
| 中途预算用完 | ≤4 | 保留最新的笔记，标记 `needs_manual_review`，不会报错中断 |

`analyze_text` 的两个 Agent 由参数传入，所以自检时可以换成“假 Agent”，不花一分钱就能测试整条流程。

In [8]:
# 组合 CheckerAgent 的审核提示词。
def make_check_prompt(paper_text: str, note: PaperNote, missing: list[str]) -> str:
    missing_text = "\n".join(f"- {item}" for item in missing) or "（全部找到）"
    return (
        "请审核以下论文精读笔记。\n\n"
        f"【论文原文】\n{paper_text}\n\n"
        f"【精读笔记】\n{note.model_dump_json(indent=2)}\n\n"
        f"【程序核对】以下 evidence 在原文中找不到：\n{missing_text}"
    )


# 组合 ReaderAgent 的修正提示词。
def make_revision_prompt(
    paper_text: str, note: PaperNote, check: CheckResult, missing: list[str]
) -> str:
    missing_text = "\n".join(f"- {item}" for item in missing) or "（全部找到）"
    return (
        "请根据审核意见修正精读笔记。仍然只能使用论文原文支持的信息，evidence 必须从原文逐字复制。\n\n"
        f"【论文原文】\n{paper_text}\n\n"
        f"【原笔记】\n{note.model_dump_json(indent=2)}\n\n"
        f"【审核意见】\n{check.model_dump_json(indent=2)}\n\n"
        f"【程序核对】以下 evidence 在原文中找不到，必须换成原文句子或删除该要点：\n{missing_text}"
    )


# 汇总需要人工复核的问题。
def collect_issues(check: CheckResult | None, missing: list[str]) -> list[str]:
    issues = []
    if check:
        issues += check.issues
        issues += [f"要点与证据不符：{item}" for item in check.unsupported_points]
        issues += [f"遗漏：{item}" for item in check.missing_points]
    issues += [f"原文中找不到证据：{item[:80]}" for item in missing]
    return issues


# 根据最终笔记和审核结论生成报告。
def make_report(
    paper: PaperText,
    note: PaperNote,
    check: CheckResult | None,
    budget: CallBudget,
    extra_issue: str | None = None,
) -> NoteReport:
    hit_rate, missing = check_evidence(note, paper.text)
    passed = check is not None and check.passed and not missing
    issues = [] if passed else collect_issues(check, missing)
    if extra_issue:
        issues.append(extra_issue)
    return NoteReport(
        source=paper.source,
        truncated=paper.truncated,
        note=note,
        evidence_hit_rate=round(hit_rate, 3),
        missing_evidence=missing,
        review_status="passed" if passed else "needs_manual_review",
        review_issues=issues,
        model_calls=budget.used,
    )


# 核心编排：生成 → 程序核对 → 审核 → 必要时修正一次。
def analyze_text(paper: PaperText, reader, checker, max_calls: int = 4) -> NoteReport:
    budget = CallBudget(maximum=max_calls)
    note = run_structured(reader, f"请为以下论文生成精读笔记：\n\n{paper.text}", PaperNote, budget)
    try:
        _, missing = check_evidence(note, paper.text)
        check = run_structured(checker, make_check_prompt(paper.text, note, missing), CheckResult, budget)
        if (check.passed and not missing) or budget.remaining < 2:
            return make_report(paper, note, check, budget)

        revision_prompt = make_revision_prompt(paper.text, note, check, missing)
        note = run_structured(reader, revision_prompt, PaperNote, budget)
        _, missing = check_evidence(note, paper.text)
        check = run_structured(checker, make_check_prompt(paper.text, note, missing), CheckResult, budget)
        return make_report(paper, note, check, budget)
    except RuntimeError as error:
        # 预算用完或格式无法修复：保留最新的笔记，交给人工复核
        return make_report(paper, note, None, budget, extra_issue=f"{error}，最新笔记未经 CheckerAgent 复核")


# 对外入口：读取论文，创建 Agent，执行完整流程。
def analyze_paper(source: str) -> NoteReport:
    paper = load_paper(source)
    suffix = f"（原文 {paper.original_chars} 字符，已截断）" if paper.truncated else ""
    print(f"已读取论文：{len(paper.text)} 字符{suffix}")
    reader, checker = build_agents()
    return analyze_text(paper, reader, checker)

## 10. 生成 Markdown 笔记

Markdown 由 Python 生成，不再让模型改写已经核对过的内容。
在原文中找不到的证据会被标上 ⚠️，读者一眼就能看出哪些内容需要自己去核实。

In [9]:
# 把报告转换成 Markdown 精读笔记。
def to_markdown(report: NoteReport) -> str:
    note = report.note
    missing = set(report.missing_evidence)
    lines = [
        f"# {note.title}",
        "",
        f"> {note.one_sentence_summary}",
        "",
        f"- **来源：** {report.source}",
        f"- **关键词：** {'、'.join(note.keywords) or '无'}",
        f"- **证据命中率：** {report.evidence_hit_rate:.0%}",
        f"- **审核状态：** `{report.review_status}`（模型调用 {report.model_calls} 次）",
    ]
    if report.truncated:
        lines.append(f"- **注意：** 原文过长，只分析了前 {MAX_PAPER_CHARS} 个字符")

    sections = [
        ("研究问题", note.research_question),
        ("核心方法", note.method),
        ("实验结论", note.experiments),
        ("局限性", note.limitations),
    ]
    for heading, claims in sections:
        lines += ["", f"## {heading}", ""]
        if not claims:
            lines.append("- 论文中未明确讨论")
            continue
        for claim in claims:
            flag = " ⚠️ 原文中未找到此句" if claim.evidence in missing else ""
            lines.append(f"- {claim.point}")
            lines.append(f"  - 原文：“{claim.evidence}”{flag}")

    if report.review_issues:
        lines += ["", "## 待人工复核", ""]
        lines += [f"- {issue}" for issue in report.review_issues]
    return "\n".join(lines) + "\n"


# 输出文件名：arXiv 编号或 PDF 文件名。
def output_stem(source: str) -> str:
    return parse_arxiv_id(source) or Path(source).stem


# 把报告保存为 JSON 和 Markdown 两个文件。
def save_report(report: NoteReport, stem: str | None = None) -> tuple[Path, Path]:
    stem = stem or output_stem(report.source)
    json_path = OUTPUT_DIR / f"{stem}_note.json"
    markdown_path = OUTPUT_DIR / f"{stem}_note.md"
    json_path.write_text(report.model_dump_json(indent=2), encoding="utf-8")
    markdown_path.write_text(to_markdown(report), encoding="utf-8")
    return json_path, markdown_path

## 11. 自检（不调用模型、不联网）

用一个“假 Agent”按顺序返回预先写好的回复，这样不需要 API Key 就能验证：
JSON 解析、证据核对、数据校验、参考文献截断，以及整条编排流程的各个分支。

In [10]:
# 假 Agent：按顺序返回预先写好的回复，并记录收到的提示词。
class FakeAgent:
    def __init__(self, responses: list[str]) -> None:
        self.responses = list(responses)
        self.prompts: list[str] = []

    def clear_history(self) -> None:
        pass

    def run(self, prompt: str) -> str:
        self.prompts.append(prompt)
        return self.responses.pop(0)


FAKE_TEXT = (
    "We propose ReAct, a general paradigm that combines reasoning and acting with language models. "
    "ReAct outperforms imitation and reinforcement learning methods on ALFWorld and WebShop. "
    "Large language models can still hallucinate facts when they reason without external knowledge."
)
fake_paper = PaperText(source="fake", text=FAKE_TEXT, original_chars=len(FAKE_TEXT), truncated=False)
GOOD = "We propose ReAct, a general paradigm that combines reasoning and acting with language models."
FAKE = "ReAct uses a novel graph neural network to plan actions."
PASS = json.dumps({"passed": True})
FAIL = json.dumps({"passed": False, "issues": ["核心方法的证据不在原文中"]})


# 生成一份假笔记，核心方法的证据由参数决定。
def fake_note(method_evidence: str) -> str:
    return json.dumps(
        {
            "title": "ReAct",
            "one_sentence_summary": "把推理和行动结合起来。",
            "keywords": ["ReAct"],
            "research_question": [
                {
                    "point": "模型只推理时会产生幻觉",
                    "evidence": "Large language models can still hallucinate facts when they reason without external knowledge.",
                }
            ],
            "method": [{"point": "结合推理与行动", "evidence": method_evidence}],
        },
        ensure_ascii=False,
    )


# 1. 带代码围栏的回复也能解析
assert parse_model_response("好的：\n```json\n" + fake_note(GOOD) + "\n```", PaperNote).title == "ReAct"

# 2. 证据核对：大小写、空格不同也算命中；编造的句子找不到
loose = GOOD.upper().replace(" ", "  ")
assert check_evidence(PaperNote.model_validate_json(fake_note(loose)), FAKE_TEXT) == (1.0, [])
assert check_evidence(PaperNote.model_validate_json(fake_note(FAKE)), FAKE_TEXT) == (0.5, [FAKE])

# 3. 太短的证据不算数；被脚注打断一次的证据算命中，但两段相隔太远不算
assert check_evidence(PaperNote.model_validate_json(fake_note("ReAct")), FAKE_TEXT)[1] == ["ReAct"]
head, tail = "We explore the fine-tuning approach on HotpotQA", "with initial promising results on this benchmark."
sentence = squash(head + " " + tail)
assert evidence_found(sentence, squash(head + " 6Human feedback is left for future work. " + tail))
assert not evidence_found(sentence, squash(head + "x" * 400 + tail))

# 4. 缺少研究问题和核心方法的笔记会被 Pydantic 拒绝
try:
    PaperNote(title="x", one_sentence_summary="x", research_question=[], method=[])
except ValidationError:
    pass
else:
    raise AssertionError("空的核心方法应被拒绝")

# 5. arXiv 编号识别、页眉页码清理、参考文献截断、文字清洗
assert parse_arxiv_id("2210.03629") == "2210.03629"
assert parse_arxiv_id("arXiv:2210.03629v3") == "2210.03629"
assert parse_arxiv_id("data/paper.pdf") is None
fake_pages = [f"Conference Header\nbody of page {n} continues\n{n}" for n in range(1, 6)]
assert strip_page_furniture(fake_pages)[2] == "body of page 3 continues"
raw = "Introduction\n" + "body text\n" * 50 + "References\n[1] Someone. 2020."
assert "Someone" not in strip_references(raw)
assert clean_text("reason-\ning  with  ﬁne-tuning") == "reasoning with fine-tuning"

# 6. 初稿编造证据 → 审核不通过 → 修正 → 通过，共 4 次调用
reader, checker = FakeAgent([fake_note(FAKE), fake_note(GOOD)]), FakeAgent([FAIL, PASS])
report = analyze_text(fake_paper, reader, checker)
assert report.review_status == "passed" and report.model_calls == 4 and report.evidence_hit_rate == 1.0
assert FAKE in checker.prompts[0].split("【程序核对】")[1]  # 程序核对结果交给了审核员

# 7. 一次通过只需 2 次调用
report = analyze_text(fake_paper, FakeAgent([fake_note(GOOD)]), FakeAgent([PASS]))
assert report.review_status == "passed" and report.model_calls == 2

# 8. 修正时 JSON 坏了 → 修复格式 → 预算用完：不报错，保留修正后的笔记并转人工复核
reader, checker = FakeAgent([fake_note(FAKE), "坏掉的回复", fake_note(GOOD)]), FakeAgent([FAIL])
report = analyze_text(fake_paper, reader, checker)
assert report.review_status == "needs_manual_review" and report.model_calls == 4
assert report.evidence_hit_rate == 1.0
assert any("未经 CheckerAgent 复核" in issue for issue in report.review_issues)

# 9. Markdown 中标出找不到的证据和未讨论的部分
report = analyze_text(fake_paper, FakeAgent([fake_note(FAKE)]), FakeAgent([FAIL]), max_calls=2)
rendered = to_markdown(report)
assert "⚠️ 原文中未找到此句" in rendered and "论文中未明确讨论" in rendered and "待人工复核" in rendered

print("自检通过：9 组")

自检通过：9 组


## 12. 运行真实论文

运行前需要在 `.env` 中填好 `LLM_MODEL_ID`、`LLM_API_KEY`、`LLM_BASE_URL`。

`PAPER_SOURCE` 可以是：
- arXiv 编号，如 `"2210.03629"`（ReAct 论文，Hello-Agents 第 4 章讲过）
- 本地 PDF 路径，如 `"data/my_paper.pdf"`

In [11]:
from IPython.display import Markdown, display

PAPER_SOURCE = "2210.03629"

report = analyze_paper(PAPER_SOURCE)
json_path, markdown_path = save_report(report)
print(f"证据命中率: {report.evidence_hit_rate:.0%}")
print(f"审核状态: {report.review_status}（模型调用 {report.model_calls} 次）")
print(f"已保存: {json_path.name}, {markdown_path.name}")
display(Markdown(to_markdown(report)))

已读取论文：41894 字符


证据命中率: 100%
审核状态: passed（模型调用 2 次）
已保存: 2210.03629_note.json, 2210.03629_note.md


# ReAct: Synergizing Reasoning and Acting in Language Models

> ReAct 提出一种提示范式，让大语言模型以交错方式生成推理轨迹与任务动作，在知识推理与交互决策任务上均优于单独推理或单独行动的基线。

- **来源：** 2210.03629
- **关键词：** ReAct、reasoning and acting、chain-of-thought、large language models、interactive decision making
- **证据命中率：** 100%
- **审核状态：** `passed`（模型调用 2 次）

## 研究问题

- 论文要解决的问题是：LLM 的推理能力（如思维链提示）与行动能力（如动作规划生成）此前主要被分开研究，尚不清楚二者能否协同结合并带来系统性收益。
  - 原文：“While large language models (LLMs) have demonstrated impressive performance across tasks in language understanding and interactive decision making, their abilities for reasoning (e.g. chain-of-thought prompting) and acting (e.g. action plan generation) have primarily been studied as separate topics.”
- 论文指出纯思维链推理是静态黑箱，不接地于外部世界，限制了模型反应式推理或更新知识的能力，容易导致事实幻觉与错误传播。
  - 原文：“However, this “chain-of-thought” reasoning is a static black box, in that the model uses its own internal representations to generate thoughts and is not grounded in the external world, which limits its ability to reason reactively or update its knowledge.”
- 论文指出此前的行动类方法通常不利用语言模型对高层目标进行抽象推理，也不维护支持行动的工作记忆。
  - 原文：“However, they do not employ language models to reason abstractly about high-level goals or maintain a working memory to support acting, barring Huang et al. (2022b) who perform a limited form of verbal reasoning to reiterate spatial facts about the current state.”

## 核心方法

- ReAct 的核心做法是让 LLM 以交错方式同时生成语言推理轨迹和任务相关动作，使推理用于创建、维护和调整高层行动计划，同时通过动作与外部环境交互以获取额外信息。
  - 原文：“ReAct prompts LLMs to generate both verbal reasoning traces and actions pertaining to a task in an interleaved manner, which allows the model to perform dynamic reasoning to create, maintain, and adjust high-level plans for acting (reason to act), while also interact with the external environments (e.g. Wikipedia) to incorporate additional information into reasoning (act to reason).”
- ReAct 将动作空间扩展为原动作空间与语言空间的并集，语言空间中的“思考”不改变外部环境、不产生观测反馈，只用于对当前上下文进行推理并更新上下文。
  - 原文：“The idea of ReAct is simple: we augment the agent’s action space to ˆA =A∪L , whereL is the space of language.”
- 论文主要采用冻结的 PaLM-540B，通过少量上下文示例提示模型同时生成领域动作和自由形式语言思考。
  - 原文：“In this paper, we mainly focus on the setup where a frozen large language model, PaLM-540B (Chowdhery et al., 2022)1, is prompted with few-shot in-context examples to generate both domain-specific actions and free-form language thoughts for task solving (Figure 1 (1d), (2b)).”
- 在推理类任务中，ReAct 交替生成思考与动作，形成多步“思考-动作-观测”轨迹；在决策类任务中，思考只需稀疏出现在轨迹中最相关的位置，由模型自行决定思考与动作的异步出现。
  - 原文：“In contrast, for decision making tasks that potentially involve a large number of actions (Figure 1(2)), thoughts only need to appear sparsely in the most relevant positions of a trajectory, so we let the language model decide the asynchronous occurrence of thoughts and actions for itself.”
- 在知识密集型任务中，ReAct 设计了一个简单的 Wikipedia 网页 API，包含 search、lookup 和 finish 三类动作以支持交互式信息检索。
  - 原文：“We design a simple Wikipedia web API with three types of actions to support interactive information retrieval: (1) search[entity], which returns the first 5 sentences from the corresponding entity wiki page if it exists, or else suggests top-5 similar entities from the Wikipedia search engine, (2) lookup[string], which would return the next sentence in the page containing string, simulating Ctrl+F functionality on the browser.”
- 论文提出将 ReAct 与 CoT-SC 结合，用启发式规则让模型在两种方法间切换，以同时利用内部知识与外部获取的信息。
  - 原文：“We therefore propose to incorporate ReAct and CoT-SC, and let the model decide when to switch to the other method based on the following heuristics: A) ReAct→ CoT-SC: when ReAct fails to return an answer within given steps, back off to CoT-SC.”
- 论文还采用类似自举的方式，用 ReAct 生成的 3000 条正确答案轨迹微调更小的 PaLM-8/62B 模型来解码轨迹。
  - 原文：“Due to the challenge of manually annotating reasoning traces and actions at scale, we consider a bootstraping approach similar to Zelikman et al. (2022), using 3,000 trajectories with correct answers generated by ReAct (also for other baselines) to finetune smaller language models (PaLM-8/62B) to decode trajectories (all thoughts, actions, observations) conditioned on input questions/claims.”

## 实验结论

- 在 HotpotQA 与 Fever 上，ReAct 一致优于仅行动的 Act 基线，说明推理对引导行动、尤其是综合最终答案具有价值。
  - 原文：“We note that ReAct is better than Act on both tasks, demonstrating the value of reasoning to guide acting, especially for synthesizing the final answer, as shown in Figure 1 (1c-d).”
- 在 Fever 上 ReAct 优于 CoT，但在 HotpotQA 上略逊于 CoT。
  - 原文：“On the other hand, ReAct outperforms CoT on Fever (60.9 vs. 56.3) and slightly lags behind CoT on HotpotQA (27.4 vs. 29.4).”
- 人工分析显示 CoT 的幻觉问题严重，其成功案例中的假阳性率高于 ReAct，且幻觉构成其主要失败模式。
  - 原文：“Hallucination is a serious problem for CoT, resulting in much higher false positive rate than ReAct (14% vs. 6%) in success mode, and make up its major failure mode (56%).”
- ReAct 的结构化约束降低了推理步骤的灵活性，导致其推理错误率高于 CoT。
  - 原文：“While interleaving reasoning, action and observation steps improves ReAct’s groundedness and trustworthiness, such a structural constraint also reduces its flexibility in formulating reasoning steps, leading to more reasoning error rate than CoT.”
- 对 ReAct 而言，通过搜索成功检索到有信息量的知识至关重要，无信息量的搜索会带偏模型推理并使其难以恢复。
  - 原文：“For ReAct, successfully retrieving informative knowledge via search is critical.”
- 在 HotpotQA 和 Fever 上最佳的提示方法分别是 ReAct→CoT-SC 与 CoT-SC→ReAct，且两种 ReAct+CoT-SC 方法在不同采样数下都显著且持续优于 CoT-SC。
  - 原文：“Also shown in Table 1, the best prompting method on HotpotQA and Fever are ReAct→ CoT-SC and CoT-SC→ ReAct respectively.”
- 微调后 ReAct 成为四种方法中最好的：PaLM-8B 微调 ReAct 超过所有 PaLM-62B 提示方法，PaLM-62B 微调 ReAct 超过所有 540B 提示方法。
  - 原文：“However, when finetuned with just 3,000 examples, ReAct becomes the best method among the four, with PaLM-8B finetuned ReAct outperforming all PaLM-62B prompting methods, and PaLM-62B finetuned ReAct outperforming all 540B prompting methods.”
- 在 ALFWorld 上，最佳 ReAct 试验平均成功率达 71%，显著优于最佳 Act（45%）与 BUTLER（37%），且 ReAct 相对 Act 的优势在六次受控试验中一致，相对提升从 33% 到 90%、平均 62%。
  - 原文：“On ALFWorld, the bestReAct trial achieves an average success rate of 71%, significantly outperforming the best Act (45%) and BUTLER (37%) trials.”
- 在 WebShop 上，ReAct 相比此前最佳成功率取得绝对 10% 的提升。
  - 原文：“With additional sparse reasoning, ReAct achieves significantly better performance, with an absolute 10% improvement over the previous best success rate.”
- 消融实验表明 ReAct 显著优于 IM 式提示（ReAct-IM），总体成功率 71% 对 53%，并在六个任务中的五个上保持优势。
  - 原文：“As can be seen in Table 3,ReAct substantially outperforms IM-style prompting ( ReAct-IM) (71 vs. 53 overall success rate), with consistent advantages on five out of six tasks.”

## 局限性

- 论文承认，尽管方法简单，但动作空间大的复杂任务需要更多示例才能学好，而这很容易超出上下文学习的输入长度限制。
  - 原文：“Despite the simplicity of our method, complex tasks with large action spaces require more demonstrations to learn well, which unfortunately can easily go beyond the input length limit of in-context learning.”
- 论文承认在 HotpotQA 上的微调只是初步的积极结果，要进一步提升性能还需要从更多高质量人工标注中学习。
  - 原文：“We explore the fine-tuning approach on HotpotQA 6Human feedback can also be incorporated in a complementary manner but we leave it for future work. with initial promising results, but learning from more high-quality human annotations will be the desiderata to further improve the performance.”
- 论文承认所有提示方法仍显著落后于领域专用最先进方法，认为用更多人工撰写数据微调可能是释放 ReAct 潜力的更好方式。
  - 原文：“As all prompting methods are still significantly far from domain-specific state-of-the-art approaches (Table 1), we believe finetuning with more human-written data might be a better way to unleash the power of ReAct.”
- 论文承认现有方法在 WebShop 上仍远不及人类专家，人类专家会进行显著更多的商品探索和查询改写，这对基于提示的方法仍然困难。
  - 原文：“However, existing methods are still far from the performance of expert humans (Table 4), who perform significantly more product explorations and query re-formulations that are still challenging for prompting-based methods.”


## 13. 总结与展望

### 实现的功能
- 支持 arXiv 编号和本地 PDF 两种输入
- 结构化精读笔记：研究问题、核心方法、实验结论、局限性
- 每条要点附英文原句证据，程序逐句核对并给出证据命中率
- ReaderAgent + CheckerAgent 协作，预算内最多修正一次
- 无需 API 的自检，覆盖编排流程的各个分支

### 遇到的挑战
- **PDF 文字不干净**：连字、断词、多余空格会让证据核对误判 → 核对前统一“压扁”两边的文字
- **句子跨页被打断**：用 ReAct 论文实测时，一句真实存在的原文被判为“找不到”，原因是它跨页，中间夹着脚注、页码和页眉 → 去掉页眉页码，并允许证据被打断一次
- **模型会“改写”证据**：即使要求逐字复制，模型仍可能润色原句 → 程序核对 + 修正提示中明确列出找不到的句子
- **论文太长**：去掉参考文献，并限制最大长度，笔记中注明是否截断

- **审核员过于挑剔**：实测中证据全部命中，审核员仍因“没写某个次要数字”判不通过 → 在提示词和字段说明里区分“核心遗漏”和“可选建议”

### 未来改进方向
- 按章节切分长论文，分段精读后再汇总
- 对找不到的证据做模糊匹配，给出原文中最接近的句子
- 支持多篇论文对比阅读
- 为扫描版 PDF 加入 OCR